# B.O.S.S. — Test YOLOv8 su recordings

**Pipeline:**
1. Estrazione ZIP e ricerca frame
2. Caricamento `best.pt` + inferenza → frame annotati in `test_boss_yolo/`
3. Distribuzione classi e confidenza
4. Preparazione Ground Truth (`BOSS_recordings.yolov8`)
5. Valutazione quantitativa contro GT (mAP, Precision, Recall)
6. Metriche per classe + grafici
7. Griglia campione frame + dashboard riepilogo

Tutti i grafici vengono salvati in `test_boss_yolo/`.

---
**Kaggle — cosa caricare:**
- **Dataset** (es. `boss-test-data`): `recordings.zip`, `BOSS_recordings.yolov8.zip`
- **Model** (es. `boss-yolo-checkpoint`): `best.pt`

Puoi caricare i ZIP direttamente senza estrarre: il notebook li estrae in `/kaggle/working/`.

In [ ]:
# ============================================================
# Cella 1 — Import librerie
# ============================================================
%pip install ultralytics pyyaml tqdm --quiet

import os
import shutil
import zipfile
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import yaml

import matplotlib.pyplot as plt

from tqdm import tqdm
from ultralytics import YOLO

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Cella 2 — Configurazione (locale + Kaggle)
# ============================================================
import torch

IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

if IS_KAGGLE:
    # ── Kaggle ──────────────────────────────────────────────
    KAGGLE_DATA_DIR  = Path("/kaggle/input/datasets/lorenzoverdura/boss-test-data")
    KAGGLE_MODEL_DIR = Path("/kaggle/input/models/lorenzoverdura/boss-yolo-checkpoint/pytorch/default/1")
    KAGGLE_GT_DIR    = KAGGLE_DATA_DIR / "BOSS_recordings.yolov8"

    BASE_DIR        = Path("/kaggle/working")
    MODEL_PATH      = KAGGLE_MODEL_DIR / "best.pt"
    RECORDINGS_ZIP  = None   # file già estratti nel dataset Kaggle
    GT_ZIP_PATH     = None   # file già estratti nel dataset Kaggle
    TEST_FRAMES_DIR = KAGGLE_DATA_DIR / "recordings/recordings/20260514_092949/frames"
else:
    # ── Locale ──────────────────────────────────────────────
    BASE_DIR        = Path("/home/lorenzoverdura8/BOSS_CODE")
    MODEL_PATH      = BASE_DIR / "best.pt"
    RECORDINGS_ZIP  = BASE_DIR / "recordings.zip"
    GT_ZIP_PATH     = BASE_DIR / "BOSS_recordings.yolov8.zip"
    TEST_FRAMES_DIR = None   # determinato in cella 3
    KAGGLE_GT_DIR   = None

# Percorsi di lavoro (scrivibili in entrambi gli ambienti)
OUTPUT_DIR   = BASE_DIR / "test_boss_yolo"
GT_EXTRACTED = BASE_DIR / "ground_truth_boss"

# Classi ostacoli B.O.S.S. — ordine identico al training
BOSS_CLASSES = [
    "person",        # 0
    "car",           # 1
    "truck",         # 2
    "bus",           # 3
    "bicycle",       # 4
    "motorcycle",    # 5
    "traffic light", # 6
    "fire hydrant",  # 7
    "stop sign",     # 8
    "bench",         # 9
    "pole",          # 10
    "chair",         # 11
    "table",         # 12
    "sofa",          # 13
    "potted plant",  # 14
    "door",          # 15
    "trash bin",     # 16
    "tree",          # 17
]
NUM_CLASSES = len(BOSS_CLASSES)  # 18

IMG_SIZE       = 640
CONF_THRESHOLD = 0.25
IOU_THRESHOLD  = 0.45
DEVICE         = "0" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GT_EXTRACTED.mkdir(parents=True, exist_ok=True)

print(f"Ambiente:       {'Kaggle' if IS_KAGGLE else 'Locale'}")
print(f"BASE_DIR:       {BASE_DIR}")
print(f"MODEL_PATH:     {MODEL_PATH}  —  esiste: {MODEL_PATH.exists()}")
if TEST_FRAMES_DIR:
    print(f"TEST_FRAMES_DIR:{TEST_FRAMES_DIR}  —  esiste: {TEST_FRAMES_DIR.exists()}")
if KAGGLE_GT_DIR:
    print(f"KAGGLE_GT_DIR:  {KAGGLE_GT_DIR}  —  esiste: {KAGGLE_GT_DIR.exists()}")
if GT_ZIP_PATH:
    print(f"GT_ZIP_PATH:    {GT_ZIP_PATH}  —  esiste: {GT_ZIP_PATH.exists()}")
print(f"OUTPUT_DIR:     {OUTPUT_DIR}")
print(f"DEVICE:         {DEVICE}")
print(f"CONF_THRESHOLD: {CONF_THRESHOLD}")
print(f"IOU_THRESHOLD:  {IOU_THRESHOLD}")

In [ ]:
# ============================================================
# Cella 3 — Estrazione ZIP e ricerca frame
# ============================================================

if IS_KAGGLE:
    # Su Kaggle i file sono già estratti nel dataset — usa il path diretto
    if not TEST_FRAMES_DIR.exists():
        raise FileNotFoundError(f"Cartella frame non trovata: {TEST_FRAMES_DIR}")
    all_frames = sorted(TEST_FRAMES_DIR.glob("*.jpg")) + sorted(TEST_FRAMES_DIR.glob("*.png"))
    print(f"TEST_FRAMES_DIR: {TEST_FRAMES_DIR}")
    print(f"Totale frame trovati: {len(all_frames)}")
else:
    # In locale: estrai recordings.zip se necessario, poi cerca le cartelle frames
    recordings_root = BASE_DIR / "recordings"

    if not recordings_root.exists() or not any(recordings_root.iterdir()):
        if not RECORDINGS_ZIP.exists():
            raise FileNotFoundError(f"ZIP non trovato: {RECORDINGS_ZIP}")
        print(f"Estrazione {RECORDINGS_ZIP} ...")
        with zipfile.ZipFile(RECORDINGS_ZIP, "r") as z:
            z.extractall(BASE_DIR)
        print(f"  -> {recordings_root}")
    else:
        print(f"OK  recordings/ già presente")

    frame_dirs = sorted([
        p for p in recordings_root.rglob("frames")
        if p.is_dir()
    ])

    all_frames = []
    for fd in frame_dirs:
        imgs = sorted(fd.glob("*.jpg")) + sorted(fd.glob("*.png"))
        all_frames.extend(imgs)
        print(f"  {fd.relative_to(BASE_DIR)}: {len(imgs)} frame")

    print(f"\nTotale frame trovati: {len(all_frames)}")

    if len(all_frames) == 0:
        raise FileNotFoundError(
            "Nessun frame trovato sotto recordings/. Verifica il contenuto del ZIP."
        )

    TEST_FRAMES_DIR = frame_dirs[0]
    print(f"TEST_FRAMES_DIR: {TEST_FRAMES_DIR}")

if len(all_frames) == 0:
    raise FileNotFoundError(f"Nessun frame trovato in {TEST_FRAMES_DIR}")

In [ ]:
# ============================================================
# Cella 4 — Caricamento modello e inferenza (Fase A)
# ============================================================
# Carica best.pt e lancia predict() su TEST_FRAMES_DIR.
# I frame annotati vengono salvati in BASE_DIR/test_boss_yolo/
# tramite project + name (percorso esatto restituito da result.save_dir).

trained_model = YOLO(str(MODEL_PATH))
print(f"Modello caricato: {MODEL_PATH}")

inference_results = trained_model.predict(
    source   = str(TEST_FRAMES_DIR),   # frame grezzi da recordings/
    conf     = CONF_THRESHOLD,         # 0.25 — allineata al training
    iou      = IOU_THRESHOLD,
    imgsz    = IMG_SIZE,
    device   = DEVICE,
    save     = True,                   # salva frame con bbox disegnate
    save_txt = True,                   # salva label YOLO .txt per ogni frame
    project  = str(BASE_DIR),          # cartella padre
    name     = "test_boss_yolo",       # → salva in BASE_DIR/test_boss_yolo/
    exist_ok = True,                   # sovrascrive run esistente
    stream   = True,                   # processa un frame alla volta (memoria contenuta)
)

# Raccoglie le predizioni in un DataFrame
all_predictions = []
PREDICT_SAVE_DIR = None

for result in inference_results:
    if PREDICT_SAVE_DIR is None:
        PREDICT_SAVE_DIR = Path(result.save_dir)  # path esatto restituito da YOLOv8

    frame_name = Path(result.path).name
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        continue

    for i in range(len(boxes)):
        cls_id = int(boxes.cls[i].item())
        conf   = float(boxes.conf[i].item())
        xyxy   = boxes.xyxy[i].cpu().numpy()
        all_predictions.append({
            "frame":      frame_name,
            "class_id":   cls_id,
            "class_name": BOSS_CLASSES[cls_id] if cls_id < NUM_CLASSES else "unknown",
            "confidence": conf,
            "x1": xyxy[0], "y1": xyxy[1], "x2": xyxy[2], "y2": xyxy[3],
        })

df_preds = pd.DataFrame(all_predictions)

print(f"\nCartella frame annotati: {PREDICT_SAVE_DIR}")
print(f"Totale predizioni:       {len(df_preds)}")
print(f"Frame con predizioni:    {df_preds['frame'].nunique() if not df_preds.empty else 0}")
if not df_preds.empty:
    print(df_preds.head(10).to_string(index=False))

df_preds.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
print(f"\nSalvato: {OUTPUT_DIR}/predictions.csv")

In [ ]:
# ============================================================
# Cella 5 — Distribuzione classi e confidenza
# ============================================================
# Grafico 1: numero rilevamenti per classe (bar chart).
# Grafico 2: distribuzione confidenza su tutte le predizioni (istogramma).

if df_preds.empty:
    print("Nessuna predizione — grafici saltati.")
else:
    # Grafico 1: rilevamenti per classe — ordine fisso = stesso del training
    class_counts = (
        df_preds["class_name"]
        .value_counts()
        .reindex(BOSS_CLASSES, fill_value=0)
    )
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = plt.cm.tab20.colors[:NUM_CLASSES]
    ax.bar(class_counts.index, class_counts.values, color=colors)
    ax.set_title("Distribuzione classi rilevate — recordings", fontsize=14)
    ax.set_xlabel("Classe")
    ax.set_ylabel("Numero rilevamenti")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_path = OUTPUT_DIR / "plot_class_distribution.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

    # Grafico 2: distribuzione confidenza
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_preds["confidence"], bins=50, color="steelblue", edgecolor="white")
    ax.axvline(
        CONF_THRESHOLD, color="red", linestyle="--",
        label=f"soglia={CONF_THRESHOLD} (allineata al training)"
    )
    ax.set_title("Distribuzione confidenza predizioni", fontsize=13)
    ax.set_xlabel("Confidenza")
    ax.set_ylabel("Frequenza")
    ax.legend()
    plt.tight_layout()
    save_path = OUTPUT_DIR / "plot_confidence_distribution.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 6 — Preparazione Ground Truth (BOSS_recordings.yolov8)
# ============================================================

if IS_KAGGLE:
    # File già estratti nel dataset — usa i path diretti (read-only)
    gt_src_base    = KAGGLE_GT_DIR
    GT_SRC_IMGS    = KAGGLE_GT_DIR / "train" / "images"
    GT_SRC_LABELS  = KAGGLE_GT_DIR / "train" / "labels"
    gt_yaml_in_path = KAGGLE_GT_DIR / "data.yaml"
    print(f"OK  GT già presente in {KAGGLE_GT_DIR}")
else:
    # Locale: estrai lo ZIP se necessario
    gt_yaml_in_path = GT_EXTRACTED / "data.yaml"
    if not gt_yaml_in_path.exists():
        if not GT_ZIP_PATH.exists():
            raise FileNotFoundError(f"ZIP ground truth non trovato: {GT_ZIP_PATH}")
        print(f"Estrazione {GT_ZIP_PATH} ...")
        with zipfile.ZipFile(GT_ZIP_PATH, "r") as z:
            z.extractall(GT_EXTRACTED)
        print(f"  -> {GT_EXTRACTED}")
    else:
        print(f"OK  ground_truth_boss/ già presente")
    GT_SRC_IMGS   = GT_EXTRACTED / "train" / "images"
    GT_SRC_LABELS = GT_EXTRACTED / "train" / "labels"

# Legge il data.yaml di Roboflow per ottenere l'ordine classi originale
with open(gt_yaml_in_path) as f:
    gt_yaml_in = yaml.safe_load(f)
RF_CLASSES = gt_yaml_in["names"]
print(f"Classi Roboflow ({len(RF_CLASSES)}): {RF_CLASSES}")

# Mappa RF_id → BOSS_id (case-insensitive)
RF_TO_BOSS_ID = {}
for rf_id, rf_name in enumerate(RF_CLASSES):
    for boss_id, boss_name in enumerate(BOSS_CLASSES):
        if rf_name.lower().strip() == boss_name.lower():
            RF_TO_BOSS_ID[rf_id] = boss_id
            break
unmapped = [RF_CLASSES[i] for i in range(len(RF_CLASSES)) if i not in RF_TO_BOSS_ID]
print(f"Mappa RF→BOSS: {RF_TO_BOSS_ID}")
print(f"Classi Roboflow scartate (non in BOSS_CLASSES): {unmapped}")


def remap_labels(src_label_dir, dst_label_dir, id_map):
    """Copia i file .txt YOLO rimappando i class ID da RF a BOSS."""
    src = Path(src_label_dir)
    dst = Path(dst_label_dir)
    dst.mkdir(parents=True, exist_ok=True)
    remapped, dropped = 0, 0
    for lf in src.glob("*.txt"):
        out_lines = []
        for line in lf.read_text().strip().splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            rf_id = int(parts[0])
            if rf_id not in id_map:
                dropped += 1
                continue
            parts[0] = str(id_map[rf_id])
            out_lines.append(" ".join(parts))
            remapped += 1
        (dst / lf.name).write_text("\n".join(out_lines))
    print(f"  Annotazioni rimappate: {remapped} | Righe scartate: {dropped}")


# Copia immagini e label rimappate in ground_truth_boss/test/ (cartella scrivibile)
GT_TEST_IMGS   = GT_EXTRACTED / "test" / "images"
GT_TEST_LABELS = GT_EXTRACTED / "test" / "labels"
GT_TEST_IMGS.mkdir(parents=True, exist_ok=True)

img_count = 0
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for img in GT_SRC_IMGS.glob(ext):
        shutil.copy(img, GT_TEST_IMGS / img.name)
        img_count += 1
print(f"Immagini copiate in test/images: {img_count}")

remap_labels(GT_SRC_LABELS, GT_TEST_LABELS, RF_TO_BOSS_ID)

# Scrive data_boss.yaml per YOLOv8 val()
gt_data_yaml = {
    "path":  str(GT_EXTRACTED),
    "train": "",
    "val":   "",
    "test":  "test/images",
    "nc":    NUM_CLASSES,
    "names": BOSS_CLASSES,
}
gt_yaml_out = GT_EXTRACTED / "data_boss.yaml"
with open(gt_yaml_out, "w") as f:
    yaml.dump(gt_data_yaml, f, default_flow_style=False, allow_unicode=True)
print(f"\ndata_boss.yaml scritto in: {gt_yaml_out}")

In [ ]:
# ============================================================
# Cella 7 — Valutazione quantitativa contro Ground Truth (Fase B)
# ============================================================
# trained_model.val() confronta le predizioni con le label in
# ground_truth_boss/test/labels/ e restituisce mAP, Precision, Recall
# per classe e aggregati, oltre a confusion matrix e curva PR.
# conf=0.001: soglia bassa per integrare tutta la curva PR → mAP corretto.

gt_val_results = trained_model.val(
    data     = str(gt_yaml_out),
    split    = "test",
    imgsz    = IMG_SIZE,
    conf     = 0.001,        # soglia bassa per calcolo mAP corretto (integra tutta la PR curve)
    iou      = IOU_THRESHOLD,
    device   = DEVICE,
    plots    = True,         # genera PR curve e confusion matrix
    project  = str(OUTPUT_DIR),
    name     = "gt_eval",
    exist_ok = True,
)

# Metriche aggregate
map50     = gt_val_results.box.map50
map5095   = gt_val_results.box.map
precision = gt_val_results.box.mp
recall    = gt_val_results.box.mr
f1_score  = 2 * (precision * recall) / (precision + recall + 1e-9)

GT_EVAL_SAVE_DIR = Path(gt_val_results.save_dir)  # path esatto per i grafici YOLOv8

print("\n=== Metriche contro Ground Truth ====")
print(f"mAP@0.5:       {map50:.4f}")
print(f"mAP@0.5:0.95:  {map5095:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"F1 Score:      {f1_score:.4f}")
print(f"Plot/CM in:    {GT_EVAL_SAVE_DIR}")

In [ ]:
# ============================================================
# Cella 8 — Metriche per classe + grafici GT
# ============================================================
# Costruisce DataFrame metriche per classe, genera AP bar chart,
# scatter P/R e mostra la confusion matrix prodotta da YOLOv8.

# Estrae metriche per classe (solo le classi presenti nel test set)
gt_class_indices = gt_val_results.box.ap_class_index.astype(int)
gt_class_names   = [BOSS_CLASSES[i] for i in gt_class_indices]

gt_per_class_ap50   = gt_val_results.box.ap50
gt_per_class_ap5095 = gt_val_results.box.ap
gt_per_class_p      = gt_val_results.box.p
gt_per_class_r      = gt_val_results.box.r
gt_per_class_f1     = 2 * (gt_per_class_p * gt_per_class_r) / (gt_per_class_p + gt_per_class_r + 1e-9)

df_metrics = pd.DataFrame({
    "Classe":      gt_class_names,
    "AP@0.5":      gt_per_class_ap50,
    "AP@0.5:0.95": gt_per_class_ap5095,
    "Precision":   gt_per_class_p,
    "Recall":      gt_per_class_r,
    "F1":          gt_per_class_f1,
})
summary_row = pd.DataFrame([{
    "Classe":      "MEDIA",
    "AP@0.5":      df_metrics["AP@0.5"].mean(),
    "AP@0.5:0.95": df_metrics["AP@0.5:0.95"].mean(),
    "Precision":   df_metrics["Precision"].mean(),
    "Recall":      df_metrics["Recall"].mean(),
    "F1":          df_metrics["F1"].mean(),
}])
df_metrics = pd.concat([df_metrics, summary_row], ignore_index=True)

float_cols = ["AP@0.5", "AP@0.5:0.95", "Precision", "Recall", "F1"]
df_display = df_metrics.copy()
df_display[float_cols] = df_display[float_cols].applymap(lambda x: f"{x:.4f}")
print("=== Metriche per Classe ===")
print(df_display.to_string(index=False))
df_metrics.to_csv(OUTPUT_DIR / "metrics_per_class_gt.csv", index=False)
print(f"\nSalvato: {OUTPUT_DIR}/metrics_per_class_gt.csv")

df_plot = df_metrics[df_metrics["Classe"] != "MEDIA"].copy()

# Grafico 1: AP@0.5 e AP@0.5:0.95 per classe
x = np.arange(len(df_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, df_plot["AP@0.5"],      width, label="AP@0.5",      color="steelblue")
bars2 = ax.bar(x + width/2, df_plot["AP@0.5:0.95"], width, label="AP@0.5:0.95", color="coral")
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.2f}",
            ha="center", va="bottom", fontsize=8)
ax.set_title("AP per classe — Ground Truth", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(df_plot["Classe"], rotation=45, ha="right")
ax.set_ylabel("AP")
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
save_path = OUTPUT_DIR / "plot_ap_per_class_gt.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"Salvato: {save_path}")

# Grafico 2: scatter Precision vs Recall per classe
fig, ax = plt.subplots(figsize=(10, 7))
for _, row in df_plot.iterrows():
    ax.scatter(row["Recall"], row["Precision"], s=100, zorder=5)
    ax.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision vs Recall per classe — Ground Truth")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_path = OUTPUT_DIR / "plot_pr_scatter_gt.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"Salvato: {save_path}")

# Grafico 3: confusion matrix generata da YOLOv8
cm_path = GT_EVAL_SAVE_DIR / "confusion_matrix_normalized.png"
if cm_path.exists():
    img_cm = Image.open(cm_path)
    plt.figure(figsize=(12, 10))
    plt.imshow(img_cm)
    plt.axis("off")
    plt.title("Confusion Matrix Normalizzata — Ground Truth")
    plt.tight_layout()
    save_path = OUTPUT_DIR / "plot_confusion_matrix_gt.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")
else:
    print(f"Confusion matrix non trovata in {cm_path}")

In [ ]:
# ============================================================
# Cella 9 — Griglia campione frame annotati
# ============================================================
# Mostra N_SAMPLES frame con bounding box disegnate da YOLOv8,
# letti da PREDICT_SAVE_DIR (cartella output dell'inferenza).

N_SAMPLES = 9

if PREDICT_SAVE_DIR is None or not PREDICT_SAVE_DIR.exists():
    print(f"Cartella inferenza non disponibile: {PREDICT_SAVE_DIR}")
    annotated_imgs = []
else:
    annotated_imgs = (
        list(PREDICT_SAVE_DIR.glob("*.jpg")) +
        list(PREDICT_SAVE_DIR.glob("*.png"))
    )

if len(annotated_imgs) == 0:
    print("Nessun frame annotato disponibile per la griglia.")
else:
    sample = random.sample(annotated_imgs, min(N_SAMPLES, len(annotated_imgs)))
    grid_size = int(np.ceil(np.sqrt(len(sample))))

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(5 * grid_size, 4 * grid_size))
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    for ax, img_path in zip(axes_flat, sample):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=7)
        ax.axis("off")
    for ax in list(axes_flat)[len(sample):]:
        ax.axis("off")

    plt.suptitle("Campione frame recordings — predizioni B.O.S.S.", fontsize=14)
    plt.tight_layout()
    save_path = OUTPUT_DIR / "plot_sample_predictions.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 10 — Dashboard riepilogo finale
# ============================================================
# Dashboard 6-pannelli che combina metriche GT aggregate, AP per classe,
# F1 per classe, scatter P/R, distribuzione classi rilevate e configurazione.

fig = plt.figure(figsize=(16, 10))
fig.suptitle("B.O.S.S. — Riepilogo Test su recordings", fontsize=16, fontweight="bold")

# Pannello 1: metriche aggregate GT come barre orizzontali
ax1 = fig.add_subplot(2, 3, 1)
metrics_summary = {
    "mAP@0.5":      map50,
    "mAP@0.5:0.95": map5095,
    "Precision":    precision,
    "Recall":       recall,
    "F1 Score":     f1_score,
}
colors_gauge = ["#2196F3", "#1976D2", "#4CAF50", "#FF9800", "#9C27B0"]
bars_h = ax1.barh(list(metrics_summary.keys()), list(metrics_summary.values()), color=colors_gauge)
for bar, val in zip(bars_h, metrics_summary.values()):
    ax1.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=10, fontweight="bold")
ax1.set_xlim(0, 1.1)
ax1.set_title("Metriche Aggregate vs GT", fontsize=11)
ax1.axvline(0.5, color="red", linestyle="--", alpha=0.5, label="soglia 0.5")
ax1.legend(fontsize=8)

# Pannello 2: AP@0.5 per classe
ax2 = fig.add_subplot(2, 3, 2)
ax2.barh(df_plot["Classe"], df_plot["AP@0.5"], color="steelblue")
ax2.set_title("AP@0.5 per Classe", fontsize=11)
ax2.set_xlim(0, 1)
ax2.axvline(map50, color="red", linestyle="--", alpha=0.7, label=f"media={map50:.2f}")
ax2.legend(fontsize=8)

# Pannello 3: F1 per classe
ax3 = fig.add_subplot(2, 3, 3)
ax3.barh(df_plot["Classe"], df_plot["F1"], color="coral")
ax3.set_title("F1 Score per Classe", fontsize=11)
ax3.set_xlim(0, 1)
ax3.axvline(f1_score, color="red", linestyle="--", alpha=0.7, label=f"media={f1_score:.2f}")
ax3.legend(fontsize=8)

# Pannello 4: scatter Precision vs Recall per classe
ax4 = fig.add_subplot(2, 3, 4)
scatter_colors = plt.cm.tab20.colors[:len(df_plot)]
for i, (_, row) in enumerate(df_plot.iterrows()):
    ax4.scatter(row["Recall"], row["Precision"],
                color=scatter_colors[i % len(scatter_colors)], s=100, zorder=5)
    ax4.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                 textcoords="offset points", xytext=(5, 3), fontsize=7)
ax4.set_xlabel("Recall")
ax4.set_ylabel("Precision")
ax4.set_title("Precision vs Recall per Classe", fontsize=11)
ax4.set_xlim(0, 1.05)
ax4.set_ylim(0, 1.05)
ax4.grid(True, alpha=0.3)

# Pannello 5: distribuzione classi rilevate sui frame recordings
ax5 = fig.add_subplot(2, 3, 5)
if not df_preds.empty:
    class_counts_test = df_preds["class_name"].value_counts()
    ax5.pie(class_counts_test.values,
            labels=class_counts_test.index,
            autopct="%1.1f%%",
            startangle=140,
            textprops={"fontsize": 7})
    ax5.set_title("Distribuzione classi — Frame recordings", fontsize=11)
else:
    ax5.text(0.5, 0.5, "Nessuna predizione", ha="center", va="center")
    ax5.axis("off")

# Pannello 6: info configurazione
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis("off")
frames_count = len(all_frames)
pred_count   = len(df_preds)
gt_img_count = len(list(GT_TEST_IMGS.glob("*.jpg"))) + len(list(GT_TEST_IMGS.glob("*.png")))
info_text = (
    f"Modello:        best.pt\n"
    f"Classi:         {NUM_CLASSES}\n"
    f"Immagine:       {IMG_SIZE}x{IMG_SIZE}\n"
    f"Conf threshold: {CONF_THRESHOLD}  (= training)\n"
    f"IoU threshold:  {IOU_THRESHOLD}\n"
    f"Device:         {DEVICE}\n\n"
    f"Frame recordings:  {frames_count}\n"
    f"Pred. totali:      {pred_count}\n"
    f"Pred/frame:        {pred_count / max(frames_count, 1):.1f}\n"
    f"GT immagini test:  {gt_img_count}\n"
    f"Output:            test_boss_yolo/\n"
)
ax6.text(0.05, 0.95, info_text, transform=ax6.transAxes,
         fontsize=10, verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.8))
ax6.set_title("Configurazione", fontsize=11)

plt.tight_layout()
save_path = OUTPUT_DIR / "plot_dashboard_riepilogo.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato: {save_path}")